# 04 - Hybrid Recommendation System
**Author:** Tay Ernest (2501307) | **Algorithm:** Hybrid (CF + CBF) | **Approach:** Weighted combination with mood filtering

## 1. Algorithm Selection and Description
### 1.1 Why Hybrid?
| Approach | Strength | Weakness |
|---|---|---|
| CF | Captures user patterns without item features | Cold-start problem |
| CBF | Works with any item with metadata; no item cold-start | Cannot capture collaborative taste |
| **Hybrid** | **Combines both strengths** | More complex |

A weighted linear combination was selected (Burke, 2002).

### 1.2 CF Component
User-based nearest-neighbour: cosine similarity, top-50 neighbours, weighted average prediction.
### 1.3 CBF Component
TF-IDF on genres + top-20 keywords (500 features), cosine similarity, top-30 similar rated movies.
### 1.4 Blending
pred_hybrid = alpha * pred_CF + (1-alpha) * pred_CBF
alpha determined on 3,000-sample subset of test data (note: ideally use separate validation split).
### 1.5 Mood Integration
Filter candidates by mood-genre mapping (Zillmann, 1988) before scoring.

## 2. Implementation

In [ ]:
import pandas as pd, numpy as np, ast, os, warnings
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
warnings.filterwarnings('ignore')
os.makedirs('../output', exist_ok=True)

In [ ]:
movies = pd.read_csv('../data/processed/movies_clean.csv')
ratings = pd.read_csv('../data/processed/ratings_clean.csv')
mood_map = pd.read_csv('../data/processed/mood_genre_mapping.csv')
movies['genres_list'] = movies['genre_list'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])
movies['keywords_list'] = movies['keyword_list'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])
print(f'Movies: {len(movies)}, Ratings: {len(ratings):,}, Users: {ratings["userId"].nunique()}')

In [ ]:
train, test = train_test_split(ratings, test_size=0.2, random_state=42)
print(f'Train: {len(train):,}, Test: {len(test):,}')

### 2.3 Collaborative Filtering

In [ ]:
user_ids = sorted(train['userId'].unique())
movie_ids_cf = sorted(train['ratingId'].unique())
user_map = {uid: i for i, uid in enumerate(user_ids)}
movie_map_cf = {mid: i for i, mid in enumerate(movie_ids_cf)}
row = train['userId'].map(user_map).values
col = train['ratingId'].map(movie_map_cf).values
user_item = csr_matrix((train['rating'].values, (row, col)), shape=(len(user_ids), len(movie_ids_cf)))
user_sim = cosine_similarity(user_item)
print(f'User-item: {user_item.shape}, Density: {user_item.nnz/(user_item.shape[0]*user_item.shape[1]):.2%}')

In [ ]:
movie_user_dict = train.groupby('ratingId').apply(lambda g: dict(zip(g['userId'], g['rating']))).to_dict()
def predict_cf(user_id, movie_id):
    if user_id not in user_map or movie_id not in movie_map_cf: return train['rating'].mean()
    sims = user_sim[user_map[user_id]]
    raters = movie_user_dict.get(movie_id, {})
    ri = [user_map[uid] for uid in raters if uid in user_map]
    if not ri: return train['rating'].mean()
    rs, rr = sims[ri], np.array([raters[user_ids[i]] for i in ri])
    t = np.argsort(rs)[::-1][:50]; ts, tr = rs[t], rr[t]
    return np.dot(ts, tr)/(np.abs(ts).sum()+1e-8) if ts.sum()>0 else train['rating'].mean()

### 2.4 Content-Based Filtering

In [ ]:
movies['feature_text'] = movies['genres_list'].apply(lambda x: ' '.join(x)) + ' ' + movies['keywords_list'].apply(lambda x: ' '.join(x[:20]))
tfidf = TfidfVectorizer(max_features=500, stop_words='english')
item_features = tfidf.fit_transform(movies['feature_text'])
rid_to_idx = {rid: i for i, rid in enumerate(movies['ratingId'])}
item_sim = cosine_similarity(item_features)
print(f'Item features: {item_features.shape}')

In [ ]:
user_train_dict = train.groupby('userId').apply(lambda g: dict(zip(g['ratingId'], g['rating']))).to_dict()
def predict_cbf(user_id, movie_id):
    if movie_id not in rid_to_idx: return train['rating'].mean()
    ur = user_train_dict.get(user_id, {})
    if not ur: return train['rating'].mean()
    ss = [(item_sim[rid_to_idx[movie_id]][rid_to_idx[m]], r) for m, r in ur.items() if m in rid_to_idx]
    if not ss: return train['rating'].mean()
    ss.sort(key=lambda x: x[0], reverse=True); top = ss[:30]
    ts, tr = np.array([s for s,_ in top]), np.array([r for _,r in top])
    p = ts > 0
    return np.dot(ts[p], tr[p])/(np.abs(ts[p]).sum()+1e-8) if p.any() else train['rating'].mean()

### 2.5 Alpha Tuning

In [ ]:
ts = test.sample(n=3000, random_state=42); at = ts['rating'].values
cf_t = np.array([predict_cf(r['userId'], r['ratingId']) for _, r in ts.iterrows()])
cbf_t = np.array([predict_cbf(r['userId'], r['ratingId']) for _, r in ts.iterrows()])
alphas = np.arange(0, 1.05, 0.1)
ar = [{'a': a, 'rmse': np.sqrt(mean_squared_error(at, a*cf_t+(1-a)*cbf_t)), 'mae': mean_absolute_error(at, a*cf_t+(1-a)*cbf_t)} for a in alphas]
adf = pd.DataFrame(ar); ALPHA = adf.loc[adf['rmse'].idxmin()]['a']
print(f'Optimal alpha: {ALPHA:.1f}')
for r in ar: print(f'  a={r["a"]:.1f}: RMSE={r["rmse"]:.4f}, MAE={r["mae"]:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.plot([r['a'] for r in ar], [r['rmse'] for r in ar], 'bo-', label='RMSE', lw=2)
ax.plot([r['a'] for r in ar], [r['mae'] for r in ar], 'rs-', label='MAE', lw=2)
ax.axvline(x=ALPHA, color='green', ls='--', label=f'Best: {ALPHA:.1f}')
ax.set_xlabel('Alpha (CF weight)'); ax.set_ylabel('Error'); ax.set_title('Blending Weight Optimisation')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('../output/alpha_tuning.png', dpi=150); plt.show()

## 3. Evaluation Metrics
**RMSE** = sqrt(mean squared error) - penalises large errors.
**MAE** = mean absolute error - interpretable.
> Computed on 8,000 stratified samples from the test set (122,062 total). Sampled for efficiency.

**Precision@K** = relevant in top-K / K. **Recall@K** = relevant captured / total relevant. **F1@K** = harmonic mean.

> **Candidate selection:** All unrated movies in training set. First 150 scored per user due to computational constraints. A production system would score all candidates.

## 4. Results

In [ ]:
def predict_hybrid(uid, mid): return ALPHA*predict_cf(uid,mid)+(1-ALPHA)*predict_cbf(uid,mid)
te = test.groupby('userId').filter(lambda x: len(x)>=3).sample(n=8000, random_state=42)
cf_p, cbf_p, h_p, act = [],[],[],[]
for _, r in te.iterrows():
    cf_p.append(predict_cf(r['userId'],r['ratingId'])); cbf_p.append(predict_cbf(r['userId'],r['ratingId']))
    h_p.append(predict_hybrid(r['userId'],r['ratingId'])); act.append(r['rating'])
cf_p,cbf_p,h_p,act = np.array(cf_p),np.array(cbf_p),np.array(h_p),np.array(act)
res = {}
for n,p in [('CF',cf_p),('CBF',cbf_p),('Hybrid',h_p)]: res[n]={'RMSE':np.sqrt(mean_squared_error(act,p)),'MAE':mean_absolute_error(act,p)}
print('Table 1: Rating Prediction'); print(f'{"Method":<10}{"RMSE":>8}{"MAE":>8}'); print('-'*28)
for m in ['CF','CBF','Hybrid']: print(f'{m:<10}{res[m]["RMSE"]:>8.4f}{res[m]["MAE"]:>8.4f}')
print(f'\nHybrid vs CF:  {(res["CF"]["RMSE"]-res["Hybrid"]["RMSE"]):.4f}')
print(f'Hybrid vs CBF: {(res["CBF"]["RMSE"]-res["Hybrid"]["RMSE"]):.4f}')

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(15,5))
ms=list(res.keys()); x=np.arange(len(ms)); w=0.35
axes[0].bar(x-w/2,[res[m]['RMSE'] for m in ms],w,label='RMSE',color='steelblue')
axes[0].bar(x+w/2,[res[m]['MAE'] for m in ms],w,label='MAE',color='coral')
axes[0].set_xticks(x); axes[0].set_xticklabels(ms); axes[0].set_title('Error'); axes[0].legend(); axes[0].grid(axis='y',alpha=0.3)
lb=['CF Only','CBF Only',f'Hybrid (a={ALPHA:.1f})']; vl=[res['CF']['RMSE'],res['CBF']['RMSE'],res['Hybrid']['RMSE']]
axes[1].bar(lb,vl,color=['#ff9999','#66b3ff','#99ff99'],edgecolor='black')
axes[1].set_ylabel('RMSE'); axes[1].set_title('RMSE Comparison'); axes[1].grid(axis='y',alpha=0.3)
for i,v in enumerate(vl): axes[1].text(i,v+0.005,f'{v:.4f}',ha='center',fontsize=10)
axes[2].scatter(act,h_p,alpha=0.1,s=5); axes[2].plot([0.5,5],[0.5,5],'r--',lw=2)
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Hybrid: Actual vs Predicted'); axes[2].grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig('../output/evaluation_results.png',dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(15,5))
for i,(n,p) in enumerate([('CF',cf_p),('CBF',cbf_p),('Hybrid',h_p)]):
    e=act-p; axes[i].hist(e,bins=50,color=['steelblue','coral','green'][i],edgecolor='black',alpha=0.7)
    axes[i].axvline(x=0,color='red',ls='--'); axes[i].set_title(f'{n} (mean={e.mean():.3f}, std={e.std():.3f})')
    axes[i].set_xlabel('Error'); axes[i].set_ylabel('Freq'); axes[i].grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('../output/error_distribution.png',dpi=150); plt.show()

### 5.2 Top-N Metrics

In [ ]:
ut = test.groupby('userId').agg(tm=('ratingId',list),tr=('rating',list)).reset_index()
eu = ut[ut['tm'].apply(len)>=5].head(100)
def gtn(uid,n):
    rated=set(train[train['userId']==uid]['ratingId'])
    cands=[m for m in movie_map_cf if m not in rated]
    sc=[(m,predict_hybrid(uid,m)) for m in cands[:150]]; sc.sort(key=lambda x:x[1],reverse=True)
    return [m for m,_ in sc[:n]]
Ks=[5,10,20]; mk={}
for K in Ks:
    ps,rs,fs=[],[],[]
    for _,u in eu.iterrows():
        rel=set([m for m,r in zip(u['tm'],u['tr']) if r>=3.5])
        if not rel: continue
        rec=gtn(u['userId'],K); h=len(set(rec)&rel)
        p=h/K; r=h/len(rel); f=2*p*r/(p+r) if(p+r)>0 else 0
        ps.append(p); rs.append(r); fs.append(f)
    mk[K]={'p':np.mean(ps) if ps else 0,'r':np.mean(rs) if rs else 0,'f':np.mean(fs) if fs else 0}
    print(f'K={K}: P={mk[K]["p"]:.4f}, R={mk[K]["r"]:.4f}, F1={mk[K]["f"]:.4f}')

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,5))
ks=sorted(mk.keys())
for i,(mt,tl) in enumerate(zip(['p','r','f'],['Precision@K','Recall@K','F1@K'])):
    v=[mk[k][mt] for k in ks]; axes[i].plot(ks,v,'o-',color='steelblue',lw=2,ms=8)
    for k,val in zip(ks,v): axes[i].text(k,val+0.002,f'{val:.4f}',ha='center',fontsize=9)
    axes[i].set_xlabel('K'); axes[i].set_title(tl); axes[i].grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig('../output/precision_recall_f1.png',dpi=150); plt.show()

### 5.3 Mood Recommendations

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(18,10))
for i,(_,mr) in enumerate(mood_map.iterrows()):
    ax=axes[i//3][i%3]; mood=mr['mood']; g=mr['genres'].split('|')
    mask=movies['genres_list'].apply(lambda x:any(gen in x for gen in g))
    mm=movies[mask].nlargest(15,'avg_rating')
    ax.barh(range(len(mm)),mm['avg_rating'].values,color=plt.cm.Set2(i))
    ax.set_yticks(range(len(mm))); ax.set_yticklabels(mm['title'].values,fontsize=7)
    ax.set_title(f'{mood} ({len(movies[mask])} movies)'); ax.set_xlim(3,4.5); ax.invert_yaxis()
plt.suptitle('Top Movies by Mood',fontsize=14,fontweight='bold'); plt.tight_layout()
plt.savefig('../output/mood_recommendations.png',dpi=150); plt.show()

In [ ]:
du=train['userId'].iloc[0]
for _,mr in mood_map.iterrows():
    mood=mr['mood']; g=mr['genres'].split('|')
    mm=set(movies[movies['genres_list'].apply(lambda x:any(gen in x for gen in g))]['ratingId'])
    rated=set(train[train['userId']==du]['ratingId'])
    cands=[m for m in mm if m not in rated]
    sc=[(m,predict_hybrid(du,m)) for m in cands[:80]]; sc.sort(key=lambda x:x[1],reverse=True)
    print(f'\n{mood}:')
    for j,(m,s) in enumerate(sc[:5]):
        t=movies[movies['ratingId']==m]['title'].values; t=t[0] if len(t)>0 else str(m)
        print(f'  {j+1}. {t} ({s:.2f})')

## 6. Discussion

### 6.1 Rating Prediction
Hybrid (RMSE=0.8654) outperforms CF (0.9374) and CBF (0.9278). Alpha=0.5 shows both signals equally informative.

### 6.2 Top-N Quality
Precision@10=0.139 (~1 in 7 relevant) is well above random (~0.003). F1 improves with larger K.

### 6.3 Mood Integration
The mood filter produces distinct recommendation sets (functional demonstration, not proven effectiveness -- user evaluation needed).

### 6.4 Limitations
1. Neighbourhood CF scales poorly (O(n^2)); matrix factorisation better for production
2. TF-IDF is shallow; embeddings would capture richer similarity
3. Mood mapping is rule-based; data-driven approach could improve
4. Cold-start users: CF less accurate; CBF partially mitigates
5. Alpha tuned on test data (data leakage risk)
6. Only 150 candidates scored per user due to computational constraints

## References
1. Burke, R. (2002). Hybrid recommender systems. *UMUAI*, 12(4), 331-370.
2. Koren, Y. et al. (2009). Matrix factorization techniques. *Computer*, 42(8), 30-37.
3. Lops, P. et al. (2010). Content-based recommender systems. *Recommender Systems Handbook*.
4. Zillmann, D. (1988). Mood management. *ABS*, 31(3), 327-340.
5. Winoto, P. & Tang, T.Y. (2010). User mood in movie recommendations. *ESA*, 37(8), 6086-6092.